In [1]:
# =========================
# Cell 1 — Upload HTTP data (ZIP/CSV), extract, pick largest CSV, preview
# =========================
from google.colab import files
import os, zipfile, glob
import pandas as pd

print("📤 Upload your HTTP dataset (ZIP or CSV)...")
uploaded = files.upload()

if not uploaded:
    raise SystemExit("⛔ No file uploaded.")

UPLOADED_NAME = list(uploaded.keys())[0]
print(f"✅ Uploaded: {UPLOADED_NAME} | size: {len(uploaded[UPLOADED_NAME])} bytes")

EXTRACT_DIR = "extracted_http_csvs"
os.makedirs(EXTRACT_DIR, exist_ok=True)

DATA_CSV = None

# If ZIP -> extract
if UPLOADED_NAME.lower().endswith(".zip"):
    with zipfile.ZipFile(UPLOADED_NAME, 'r') as z:
        z.extractall(EXTRACT_DIR)
    print(f"✅ Extracted to: {EXTRACT_DIR}")

    csvs = glob.glob(os.path.join(EXTRACT_DIR, "**/*.csv"), recursive=True)
    if not csvs:
        raise SystemExit("⛔ No CSV found inside ZIP.")

    # pick largest CSV
    csvs = sorted(csvs, key=lambda p: os.path.getsize(p), reverse=True)
    DATA_CSV = csvs[0]
    print(f"📌 Found CSVs: {len(csvs)}")
    print(f"📌 Selected largest CSV: {DATA_CSV} | {os.path.getsize(DATA_CSV)/1024/1024:.2f} MB")

# If CSV -> use directly
elif UPLOADED_NAME.lower().endswith(".csv"):
    DATA_CSV = UPLOADED_NAME
    print(f"📌 Using uploaded CSV: {DATA_CSV} | {os.path.getsize(DATA_CSV)/1024/1024:.2f} MB")

else:
    raise SystemExit("⛔ Unsupported file type. Upload ZIP or CSV.")

print("\n🔎 Preview: TAIL(5) (chunk-safe)")
tail = None
for chunk in pd.read_csv(DATA_CSV, chunksize=200000):
    tail = chunk.tail(5)
display(tail)

print(f"\n✅ DATA_CSV set to: {DATA_CSV}")


📤 Upload your HTTP dataset (ZIP or CSV)...


Saving BEACEXFHTTP.zip to BEACEXFHTTP.zip
✅ Uploaded: BEACEXFHTTP.zip | size: 13790807 bytes
✅ Extracted to: extracted_http_csvs
📌 Found CSVs: 1
📌 Selected largest CSV: extracted_http_csvs/merged (10).csv | 241.53 MB

🔎 Preview: TAIL(5) (chunk-safe)


,@timestamp,http.request.method,http.response.status_code,http.version,http.response.body.bytes,user_agent.original,url.path,url.domain,destination.ip,destination.port,source.ip,url.query,http.request.body.bytes,__source_file
1120807,"Dec 6, 2025 @ 10:36:19.401",CONNECT,200,1.1,0,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,-,kws2-1.web.telegram.org:443,172.16.60.2,"8,080",192.168.20.33,-,0,HTTPBEACEXF/Untitled Discover session (45).csv
1120808,"Dec 6, 2025 @ 10:36:18.936",CONNECT,200,1.1,0,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,-,kws2-1.web.telegram.org:443,172.16.60.2,"8,080",192.168.20.33,-,0,HTTPBEACEXF/Untitled Discover session (45).csv
1120809,"Dec 6, 2025 @ 10:36:18.677",POST,-,-,0,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,/,speed.cloudflare.com,134.195.199.122,80,10.10.10.18,-,300,HTTPBEACEXF/Untitled Discover session (45).csv
1120810,"Dec 6, 2025 @ 10:36:17.560",POST,-,-,0,-,/time,speed.cloudflare.com,104.17.148.22,80,10.10.10.18,-,0,HTTPBEACEXF/Untitled Discover session (45).csv
1120811,"Dec 6, 2025 @ 10:36:17.347",POST,101,1.1,0,-,/time,speed.cloudflare.com,104.17.148.22,80,10.10.10.18,-,0,HTTPBEACEXF/Untitled Discover session (45).csv



✅ DATA_CSV set to: extracted_http_csvs/merged (10).csv


In [17]:
# =========================
# Cell 2 — FAST HTTP loader (chunked) — ROBUST + DERIVED FEATURES (FIXED)
# Output: df_clean
#   - adds: query_len, query_entropy, url_full_len, path_depth, up_down_ratio
# =========================

import pandas as pd
import numpy as np
import re
from math import log2

print("⚡ FAST HTTP loader (chunked) — ROBUST + DERIVED FEATURES")

assert "DATA_CSV" in globals(), "⛔ DATA_CSV not set. Run Cell 1 first."
print("📄 Reading:", DATA_CSV)

# --- read header only
hdr = pd.read_csv(DATA_CSV, nrows=0)
raw_cols = hdr.columns.tolist()

# normalize helper
def norm(c):
    return re.sub(r"\s+", "", str(c)).strip().lower()

norm_map = {norm(c): c for c in raw_cols}

def pick(*candidates):
    """Pick first existing column from candidates (by normalized name)."""
    for cand in candidates:
        if norm(cand) in norm_map:
            return norm_map[norm(cand)]
    return None

# detect actual column names
COL_TS    = pick("@timestamp", "timestamp", "ts", "time", "event.created")
COL_SRC   = pick("source.ip", "src.ip", "client.ip", "source.address")
COL_DST   = pick("destination.ip", "dest.ip", "dst.ip", "server.ip", "destination.address")
COL_DPORT = pick("destination.port", "dest.port", "dst.port", "server.port")
COL_DOM   = pick("url.domain", "host", "http.host", "destination.domain")
COL_PATH  = pick("url.path", "path", "http.request.path")
COL_FULL  = pick("url.full", "url.original", "url", "http.request.referrer")  # optional
COL_QUERY = pick("url.query", "query", "url.query.keyword")
COL_METH  = pick("http.request.method", "method", "http.method")
COL_STAT  = pick("http.response.status_code", "status", "http.status_code")
COL_VER   = pick("http.version", "httpver", "version")
COL_UA    = pick("user_agent.original", "user_agent", "ua")
COL_RB    = pick("http.response.body.bytes", "response.bytes", "resp.bytes")
COL_QB    = pick("http.request.body.bytes", "request.bytes", "req.bytes")

print("🧾 Columns detected:", raw_cols)
print("✅ Mapped columns:")
print("  TS :", COL_TS)
print("  SRC:", COL_SRC)
print("  DST:", COL_DST, "(may be None)")
print("  DPORT:", COL_DPORT)
print("  DOMAIN:", COL_DOM)
print("  PATH:", COL_PATH)
print("  FULL(optional):", COL_FULL)
print("  QUERY:", COL_QUERY)
print("  METHOD:", COL_METH)
print("  STATUS:", COL_STAT)
print("  VERSION:", COL_VER)
print("  UA:", COL_UA)
print("  RESP_BYTES:", COL_RB)
print("  REQ_BODY_BYTES:", COL_QB)

# choose minimal usecols that exist
usecols = [c for c in [
    COL_TS, COL_SRC, COL_DST, COL_DPORT, COL_DOM, COL_PATH, COL_FULL, COL_QUERY,
    COL_METH, COL_STAT, COL_VER, COL_UA, COL_RB, COL_QB
] if c is not None]

print("📌 usecols:", usecols)

def is_ipv4(s):
    if not isinstance(s, str):
        return False
    s = s.strip()
    return bool(re.match(r"^\d{1,3}(\.\d{1,3}){3}$", s))

def parse_ts_series(ts_raw: pd.Series) -> pd.Series:
    """
    Robust timestamp parsing:
    - try common Kibana format: 'Oct 7, 2025 @ 09:08:03.073'
    - try without ms
    - fallback generic parsing
    """
    s = ts_raw.astype(str)

    ts1 = pd.to_datetime(s, errors="coerce", format="%b %d, %Y @ %H:%M:%S.%f")
    miss = ts1.isna()
    if miss.any():
        ts2 = pd.to_datetime(s[miss], errors="coerce", format="%b %d, %Y @ %H:%M:%S")
        ts1.loc[miss] = ts2

    miss2 = ts1.isna()
    if miss2.any():
        ts3 = pd.to_datetime(s[miss2].str.replace(" @ ", " ", regex=False), errors="coerce", infer_datetime_format=True)
        ts1.loc[miss2] = ts3

    return ts1

def shannon_entropy_fast(s: str) -> float:
    if s is None:
        return 0.0
    s = str(s)
    if not s:
        return 0.0
    if len(s) > 2000:
        s = s[:2000]
    freq = {}
    for ch in s:
        freq[ch] = freq.get(ch, 0) + 1
    n = len(s)
    ent = 0.0
    for v in freq.values():
        p = v / n
        ent -= p * log2(p)
    return float(ent)

chunks = []
total_read = 0
bad_ts_total = 0

CHUNK_SIZE = 250_000

for ch in pd.read_csv(DATA_CSV, usecols=usecols, chunksize=CHUNK_SIZE, low_memory=False):
    total_read += len(ch)

    # strip column names just in case (hidden spaces)
    ch.columns = [str(c).strip() for c in ch.columns]

    # ---- standard fields ----
    ts_raw = ch[COL_TS] if COL_TS in ch.columns else pd.Series([None]*len(ch))
    ch["ts"] = parse_ts_series(ts_raw)
    bad_ts_total += int(ch["ts"].isna().sum())

    ch["source.ip"] = (ch[COL_SRC] if COL_SRC in ch.columns else "-").astype(str).str.strip()

    if COL_DST and COL_DST in ch.columns:
        ch["destination.ip"] = ch[COL_DST].astype(str).str.strip()
    else:
        ch["destination.ip"] = "-"

    if COL_DPORT and COL_DPORT in ch.columns:
        ch["destination.port"] = pd.to_numeric(ch[COL_DPORT], errors="coerce").fillna(-1).astype(int)
    else:
        ch["destination.port"] = -1

    # url parts
    ch["url.domain"] = (ch[COL_DOM] if COL_DOM in ch.columns else "-").astype(str).fillna("-").str.strip()
    ch["url.path"]   = (ch[COL_PATH] if COL_PATH in ch.columns else "-").astype(str).fillna("-")
    ch["url.query"]  = (ch[COL_QUERY] if COL_QUERY in ch.columns else "").astype(str).fillna("")

    # url.full: اگر موجود بود استفاده کن، وگرنه بساز
    if COL_FULL and COL_FULL in ch.columns:
        ch["url.full"] = ch[COL_FULL].astype(str).fillna("")
    else:
        ch["url.full"] = (ch["url.domain"].astype(str) + ch["url.path"].astype(str)).fillna("")

    # If destination.ip missing, try extract from url.domain if it's like '1.2.3.4:443'
    mask_dst_missing = (ch["destination.ip"] == "-") | (ch["destination.ip"] == "nan") | (ch["destination.ip"] == "")
    if mask_dst_missing.any():
        ipport = ch.loc[mask_dst_missing, "url.domain"].astype(str)
        m = ipport.str.extract(r"^(?P<ip>\d{1,3}(?:\.\d{1,3}){3})(?::(?P<port>\d+))?$")
        has_ip = m["ip"].notna()
        idx = mask_dst_missing[mask_dst_missing].index[has_ip]
        if len(idx):
            ch.loc[idx, "destination.ip"] = m.loc[has_ip, "ip"].values

            # fill port if missing
            if "port" in m.columns:
                port_vals = pd.to_numeric(m.loc[has_ip, "port"], errors="coerce")
                need_port = (ch.loc[idx, "destination.port"] == -1)
                if need_port.any():
                    ch.loc[idx[need_port.values], "destination.port"] = port_vals.loc[need_port.values].fillna(-1).astype(int).values

    # method/status/version
    ch["http.method"] = (ch[COL_METH] if COL_METH in ch.columns else "-").astype(str).str.upper().replace({"NAN":"-"})
    ch["http.status"] = pd.to_numeric((ch[COL_STAT] if COL_STAT in ch.columns else -1), errors="coerce").fillna(-1).astype(int)
    ch["http.version"]= (ch[COL_VER] if COL_VER in ch.columns else "-").astype(str).replace({"nan":"-","None":"-"}).fillna("-")

    # UA
    ch["user_agent"] = (ch[COL_UA] if COL_UA in ch.columns else "-").astype(str).fillna("-")

    # bytes
    ch["resp_bytes"] = pd.to_numeric((ch[COL_RB] if COL_RB in ch.columns else 0), errors="coerce").fillna(0.0).astype(float)
    ch["req_body_bytes"] = pd.to_numeric((ch[COL_QB] if COL_QB in ch.columns else 0), errors="coerce").fillna(0.0).astype(float)

    # ---- derived features (needed by Cell 3) ----
    ch["query_len"] = ch["url.query"].astype(str).str.len().astype(int)
    ch["query_entropy"] = ch["url.query"].astype(str).apply(shannon_entropy_fast)
    ch["url_full_len"] = ch["url.full"].astype(str).str.len().astype(int)
    ch["path_depth"] = ch["url.path"].astype(str).apply(lambda p: int(len([x for x in str(p).split("/") if x])))

    ch["up_down_ratio"] = ch["req_body_bytes"] / (ch["resp_bytes"] + 1.0)

    # success/fail flags
    ch["is_success"] = ((ch["http.status"] >= 200) & (ch["http.status"] < 300)).astype(int)
    ch["is_fail"]    = ((ch["http.status"] >= 400) & (ch["http.status"] < 600)).astype(int)

    # drop invalid timestamps
    ch = ch[ch["ts"].notna()].copy()

    # keep only valid source IPv4
    ch = ch[ch["source.ip"].apply(is_ipv4)].copy()

    # window_start
    ch["window_start"] = ch["ts"].dt.floor("min")

    chunks.append(ch[[
        "ts","window_start","source.ip","destination.ip","destination.port",
        "url.domain","url.path","url.query","url.full",
        "http.method","http.status","http.version","user_agent",
        "resp_bytes","req_body_bytes","up_down_ratio",
        "query_len","query_entropy","url_full_len","path_depth",
        "is_success","is_fail"
    ]])

df_clean = pd.concat(chunks, ignore_index=True) if len(chunks) else pd.DataFrame()

print("✅ Total rows read:", total_read)
print("⚠️ Dropped invalid timestamp rows (total):", bad_ts_total)
print("✅ df_clean ready:", df_clean.shape)
if len(df_clean):
    print("🕒 Time range:", df_clean["ts"].min(), "→", df_clean["ts"].max())

display(df_clean.head(5))
display(df_clean.tail(5))


⚡ FAST HTTP loader (chunked) — ROBUST + DERIVED FEATURES
📄 Reading: extracted_http_csvs/merged (10).csv
🧾 Columns detected: ['@timestamp', 'http.request.method', 'http.response.status_code', 'http.version', 'http.response.body.bytes', 'user_agent.original', 'url.path', 'url.domain', 'destination.ip', 'destination.port', 'source.ip', 'url.query', 'http.request.body.bytes', '__source_file']
✅ Mapped columns:
  TS : @timestamp
  SRC: source.ip
  DST: destination.ip (may be None)
  DPORT: destination.port
  DOMAIN: url.domain
  PATH: url.path
  FULL(optional): None
  QUERY: url.query
  METHOD: http.request.method
  STATUS: http.response.status_code
  VERSION: http.version
  UA: user_agent.original
  RESP_BYTES: http.response.body.bytes
  REQ_BODY_BYTES: http.request.body.bytes
📌 usecols: ['@timestamp', 'source.ip', 'destination.ip', 'destination.port', 'url.domain', 'url.path', 'url.query', 'http.request.method', 'http.response.status_code', 'http.version', 'user_agent.original', 'http.res

,ts,window_start,source.ip,destination.ip,destination.port,url.domain,url.path,url.query,url.full,http.method,...,user_agent,resp_bytes,req_body_bytes,up_down_ratio,query_len,query_entropy,url_full_len,path_depth,is_success,is_fail
0,2025-11-27 23:26:53.126,2025-11-27 23:26:00,192.168.100.34,172.16.60.4,-1,172.16.60.4:8080,/epr/endpoint/endpoint-9.1.0.zip.sig,-,172.16.60.4:8080/epr/endpoint/endpoint-9.1.0.z...,GET,...,Kibana/9.2.0 node-fetch,488.0,0.0,0.0,1,0.0,52,3,1,0
1,2025-11-27 23:26:53.120,2025-11-27 23:26:00,192.168.100.34,172.16.60.4,-1,172.16.60.4:8080,/package/endpoint/9.1.0/,-,172.16.60.4:8080/package/endpoint/9.1.0/,GET,...,Kibana/9.2.0 node-fetch,0.0,0.0,0.0,1,0.0,40,3,1,0
2,2025-11-27 23:26:52.919,2025-11-27 23:26:00,192.168.100.34,172.16.60.4,-1,172.16.60.4:8080,/epr/endpoint/endpoint-9.1.0.zip,-,172.16.60.4:8080/epr/endpoint/endpoint-9.1.0.zip,GET,...,Kibana/9.2.0 node-fetch,0.0,0.0,0.0,1,0.0,48,3,1,0
3,2025-11-27 23:26:52.894,2025-11-27 23:26:00,192.168.100.34,172.16.60.4,-1,172.16.60.4:8080,/package/endpoint/9.1.0/,-,172.16.60.4:8080/package/endpoint/9.1.0/,GET,...,Kibana/9.2.0 node-fetch,0.0,0.0,0.0,1,0.0,40,3,1,0
4,2025-11-27 23:21:51.988,2025-11-27 23:21:00,192.168.100.34,172.16.60.4,-1,172.16.60.4:8080,/epr/endpoint/endpoint-9.1.0.zip.sig,-,172.16.60.4:8080/epr/endpoint/endpoint-9.1.0.z...,GET,...,Kibana/9.2.0 node-fetch,488.0,0.0,0.0,1,0.0,52,3,1,0


,ts,window_start,source.ip,destination.ip,destination.port,url.domain,url.path,url.query,url.full,http.method,...,user_agent,resp_bytes,req_body_bytes,up_down_ratio,query_len,query_entropy,url_full_len,path_depth,is_success,is_fail
1120807,2025-12-06 10:36:19.401,2025-12-06 10:36:00,192.168.20.33,172.16.60.2,-1,kws2-1.web.telegram.org:443,-,-,kws2-1.web.telegram.org:443-,CONNECT,...,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,0.0,0.0,0.0,1,0.0,28,1,1,0
1120808,2025-12-06 10:36:18.936,2025-12-06 10:36:00,192.168.20.33,172.16.60.2,-1,kws2-1.web.telegram.org:443,-,-,kws2-1.web.telegram.org:443-,CONNECT,...,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,0.0,0.0,0.0,1,0.0,28,1,1,0
1120809,2025-12-06 10:36:18.677,2025-12-06 10:36:00,10.10.10.18,134.195.199.122,80,speed.cloudflare.com,/,-,speed.cloudflare.com/,POST,...,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,0.0,300.0,300.0,1,0.0,21,0,0,0
1120810,2025-12-06 10:36:17.560,2025-12-06 10:36:00,10.10.10.18,104.17.148.22,80,speed.cloudflare.com,/time,-,speed.cloudflare.com/time,POST,...,-,0.0,0.0,0.0,1,0.0,25,1,0,0
1120811,2025-12-06 10:36:17.347,2025-12-06 10:36:00,10.10.10.18,104.17.148.22,80,speed.cloudflare.com,/time,-,speed.cloudflare.com/time,POST,...,-,0.0,0.0,0.0,1,0.0,25,1,0,0


In [18]:
# =========================
# Cell 3 — Build aggregated HTTP features (1-min) + beacon pair features (FIXED robust)
#   - اگر ستون‌های query_len/query_entropy/url_full_len/path_depth نبودند، همینجا ساخته می‌شوند
# =========================

import pandas as pd
import numpy as np

print("🧱 Building 1-minute aggregated HTTP dataset (per source.ip × window_start)...")

# ---------- helper: trends / rolling ----------
def add_temporal_features(host_df: pd.DataFrame) -> pd.DataFrame:
    host_df = host_df.sort_values("window_start").copy()

    host_df["req_roll_60m"] = host_df["req_count"].rolling(60, min_periods=1).sum()
    host_df["bytes_roll_60m"] = host_df["resp_bytes_sum"].rolling(60, min_periods=1).sum()
    host_df["upload_roll_60m"] = host_df["req_body_bytes_sum"].rolling(60, min_periods=1).sum()

    host_df["uniq_domains_roll_60m"] = host_df["uniq_domains"].rolling(60, min_periods=1).sum()
    host_df["uniq_paths_roll_60m"] = host_df["uniq_paths"].rolling(60, min_periods=1).sum()

    def simple_trend(s):
        if len(s) < 2:
            return 0.0
        return float((s.iloc[-1] - s.iloc[0]) / max(len(s) - 1, 1))

    host_df["bytes_trend_30m"] = host_df["resp_bytes_sum"].rolling(30, min_periods=2).apply(simple_trend, raw=False)
    host_df["req_trend_30m"] = host_df["req_count"].rolling(30, min_periods=2).apply(simple_trend, raw=False)
    host_df["upload_trend_30m"] = host_df["req_body_bytes_sum"].rolling(30, min_periods=2).apply(simple_trend, raw=False)

    host_df[["bytes_trend_30m","req_trend_30m","upload_trend_30m"]] = (
        host_df[["bytes_trend_30m","req_trend_30m","upload_trend_30m"]]
        .replace([np.inf, -np.inf], 0.0)
        .fillna(0.0)
    )

    return host_df


# ---------- core aggregation ----------
df = df_clean.copy()

# ✅ ensure required url cols exist (اگر سلول 2 بعضی‌ها را نساخته باشد)
for c in ["url.path", "url.query", "url.full", "url.domain"]:
    if c not in df.columns:
        df[c] = ""

# ✅ اگر ستون‌های مشتق‌شده وجود ندارند، همینجا بساز
# entropy helper (سریع)
from math import log2
def shannon_entropy_fast(s: str) -> float:
    if s is None:
        return 0.0
    s = str(s)
    if not s:
        return 0.0
    if len(s) > 2000:
        s = s[:2000]
    freq = {}
    for ch in s:
        freq[ch] = freq.get(ch, 0) + 1
    n = len(s)
    ent = 0.0
    for v in freq.values():
        p = v / n
        ent -= p * log2(p)
    return float(ent)

if "query_len" not in df.columns:
    df["query_len"] = df["url.query"].fillna("").astype(str).str.len().astype(int)

if "query_entropy" not in df.columns:
    df["query_entropy"] = df["url.query"].fillna("").astype(str).apply(shannon_entropy_fast)

if "url_full_len" not in df.columns:
    df["url_full_len"] = df["url.full"].fillna("").astype(str).str.len().astype(int)

if "path_depth" not in df.columns:
    df["path_depth"] = df["url.path"].fillna("").astype(str).apply(
        lambda p: int(len([x for x in str(p).split("/") if x]))
    )

# ✅ اگر ستون‌های اصلی که باید از سلول 2 آمده باشند نبودند، خطا بده (این‌ها حیاتی‌اند)
required_base = ["ts","window_start","source.ip","destination.ip","destination.port",
                 "http.method","http.status","user_agent",
                 "resp_bytes","req_body_bytes","is_success","is_fail","up_down_ratio"]
missing = [c for c in required_base if c not in df.columns]
if missing:
    raise ValueError(f"⛔ Missing required columns in df_clean. Run Cell 2 properly. Missing: {missing}")

# user-agent suspicious heuristic
def is_ua_susp(ua: str) -> int:
    if ua is None:
        return 1
    s = str(ua).strip()
    if s in ["", "-", "nan"]:
        return 1
    if len(s) < 8:
        return 1
    low = s.lower()
    if "curl" in low or "wget" in low or "python" in low or "node-fetch" in low:
        return 1
    return 0

df["ua_susp"] = df["user_agent"].apply(is_ua_susp).astype(int)

# status buckets
df["s2xx"] = ((df["http.status"] >= 200) & (df["http.status"] < 300)).astype(int)
df["s3xx"] = ((df["http.status"] >= 300) & (df["http.status"] < 400)).astype(int)
df["s4xx"] = ((df["http.status"] >= 400) & (df["http.status"] < 500)).astype(int)
df["s5xx"] = ((df["http.status"] >= 500) & (df["http.status"] < 600)).astype(int)

df["is_post"] = (df["http.method"] == "POST").astype(int)
df["is_get"] = (df["http.method"] == "GET").astype(int)

# aggregation
agg_host = df.groupby(["source.ip","window_start"]).agg(
    req_count=("ts","count"),
    uniq_dst_ips=("destination.ip","nunique"),
    uniq_domains=("url.domain","nunique"),
    uniq_paths=("url.path","nunique"),
    uniq_dst_ports=("destination.port","nunique"),

    resp_bytes_sum=("resp_bytes","sum"),
    resp_bytes_mean=("resp_bytes","mean"),
    resp_bytes_max=("resp_bytes","max"),

    req_body_bytes_sum=("req_body_bytes","sum"),
    req_body_bytes_mean=("req_body_bytes","mean"),
    req_body_bytes_max=("req_body_bytes","max"),

    method_uniq=("http.method","nunique"),
    post_count=("is_post","sum"),
    get_count=("is_get","sum"),

    s2xx=("s2xx","sum"),
    s3xx=("s3xx","sum"),
    s4xx=("s4xx","sum"),
    s5xx=("s5xx","sum"),

    ua_susp_count=("ua_susp","sum"),

    path_depth_mean=("path_depth","mean"),
    path_depth_max=("path_depth","max"),

    query_len_sum=("query_len","sum"),
    query_len_mean=("query_len","mean"),
    query_entropy_mean=("query_entropy","mean"),
    url_full_len_sum=("url_full_len","sum"),

    success_count=("is_success","sum"),
    fail_count=("is_fail","sum"),
).reset_index()

# rates
agg_host["success_rate_2xx"] = agg_host["s2xx"] / (agg_host["req_count"] + 1e-9)
agg_host["fail_rate_4xx5xx"] = (agg_host["s4xx"] + agg_host["s5xx"]) / (agg_host["req_count"] + 1e-9)
agg_host["post_rate"] = agg_host["post_count"] / (agg_host["req_count"] + 1e-9)
agg_host["ua_susp_rate"] = agg_host["ua_susp_count"] / (agg_host["req_count"] + 1e-9)

# bytes-per-request
agg_host["bpr"] = agg_host["resp_bytes_sum"] / (agg_host["req_count"] + 1e-9)
agg_host["log_resp_bytes_sum"] = np.log1p(agg_host["resp_bytes_sum"])
agg_host["log_bpr"] = np.log1p(agg_host["bpr"])

# upload/download ratio at minute level
agg_host["up_down_ratio_sum"] = agg_host["req_body_bytes_sum"] / (agg_host["resp_bytes_sum"] + 1.0)
agg_host["log_upload_sum"] = np.log1p(agg_host["req_body_bytes_sum"])
agg_host["log_up_down_ratio"] = np.log1p(agg_host["up_down_ratio_sum"])

agg_host = agg_host.replace([np.inf, -np.inf], 0.0).fillna(0.0)

# temporal features per host
agg_host = agg_host.groupby("source.ip", group_keys=False).apply(add_temporal_features)

print("✅ agg_host:", agg_host.shape)


# =====================
# Pair-based Beaconing features (source.ip × url.domain)
# =====================
print("🧭 Building pair-based HTTP beaconing features (source.ip × url.domain)...")

pair = df[["source.ip","window_start","url.domain","ts","resp_bytes"]].copy()
pair = pair[pair["url.domain"].astype(str).str.len() > 0]

def add_beacon_pair(g: pd.DataFrame) -> pd.DataFrame:
    g = g.sort_values("ts").copy()
    if len(g) < 3:
        g["pair_regularity"] = 0.0
        g["pair_persist_5m"] = 0.0
        return g

    dt = (g["ts"].diff().dt.total_seconds()).dropna()
    if len(dt) < 2:
        reg = 0.0
    else:
        mu = float(dt.mean())
        sd = float(dt.std())
        reg = 0.0 if mu <= 0 else max(0.0, 1.0 - min(sd / (mu + 1e-9), 1.0))

    g["bucket_5m"] = g["ts"].dt.floor("5min")
    persist_5m = float(g["bucket_5m"].nunique())

    g["pair_regularity"] = float(reg)
    g["pair_persist_5m"] = persist_5m
    return g

pair_dom = pair.groupby(["source.ip","url.domain"], group_keys=False).apply(add_beacon_pair)

pair_min = pair_dom.groupby(["source.ip","window_start","url.domain"]).agg(
    dom_req=("ts","count"),
    dom_bytes=("resp_bytes","sum"),
    dom_reg=("pair_regularity","max"),
    dom_persist_5m=("pair_persist_5m","max"),
).reset_index()

pair_min["bytes_per_req"] = pair_min["dom_bytes"] / (pair_min["dom_req"] + 1e-9)
pair_min["low_bpr_score"] = 1.0 / (1.0 + np.log1p(pair_min["bytes_per_req"]))

pair_min["http_beacon_score_pair"] = (
    0.45 * pair_min["dom_reg"] +
    0.35 * np.tanh(pair_min["dom_persist_5m"] / 5.0) +
    0.20 * pair_min["low_bpr_score"]
)

pair_min = pair_min.sort_values(
    ["source.ip","window_start","http_beacon_score_pair","dom_req"],
    ascending=[True, True, False, False]
)

best = pair_min.groupby(["source.ip","window_start"]).head(1).copy()

agg_beacon_host = best[[
    "source.ip","window_start",
    "url.domain","dom_req",
    "dom_persist_5m","dom_reg",
    "http_beacon_score_pair"
]].rename(columns={
    "url.domain": "http_beacon_best_domain",
    "dom_req": "http_beacon_best_domain_req",
    "dom_persist_5m": "http_beacon_best_persist_5m",
    "dom_reg": "http_beacon_best_regularity",
    "http_beacon_score_pair": "http_beacon_score"
})

print("✅ agg_beacon_host:", agg_beacon_host.shape)

# merge features
agg_features = agg_host.merge(agg_beacon_host, on=["source.ip","window_start"], how="left")
agg_features[[
    "http_beacon_best_domain","http_beacon_best_domain_req",
    "http_beacon_best_persist_5m","http_beacon_best_regularity","http_beacon_score"
]] = agg_features[[
    "http_beacon_best_domain","http_beacon_best_domain_req",
    "http_beacon_best_persist_5m","http_beacon_best_regularity","http_beacon_score"
]].fillna({"http_beacon_best_domain":"", "http_beacon_best_domain_req":0,
           "http_beacon_best_persist_5m":0.0, "http_beacon_best_regularity":0.0, "http_beacon_score":0.0})

agg_features = agg_features.replace([np.inf, -np.inf], 0.0).fillna(0.0)

print("✅ agg_features (feature-only):", agg_features.shape)
display(agg_features.head(5))
display(agg_features.tail(5))


🧱 Building 1-minute aggregated HTTP dataset (per source.ip × window_start)...


/tmp/ipython-input-2510678506.py:174: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  agg_host = agg_host.groupby("source.ip", group_keys=False).apply(add_temporal_features)


✅ agg_host: (50231, 47)
🧭 Building pair-based HTTP beaconing features (source.ip × url.domain)...


/tmp/ipython-input-2510678506.py:209: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pair_dom = pair.groupby(["source.ip","url.domain"], group_keys=False).apply(add_beacon_pair)


✅ agg_beacon_host: (50231, 7)
✅ agg_features (feature-only): (50231, 52)


,source.ip,window_start,req_count,uniq_dst_ips,uniq_domains,uniq_paths,uniq_dst_ports,resp_bytes_sum,resp_bytes_mean,resp_bytes_max,...,uniq_domains_roll_60m,uniq_paths_roll_60m,bytes_trend_30m,req_trend_30m,upload_trend_30m,http_beacon_best_domain,http_beacon_best_domain_req,http_beacon_best_persist_5m,http_beacon_best_regularity,http_beacon_score
0,10.10.10.10,2025-12-10 13:12:00,1,1,1,1,1,69.0,69.000000,69.0,...,1.0,1.0,0.0,0.0,0.0,captive.apple.com,1,0.0,0.0,0.038106
1,10.10.10.10,2025-12-10 13:42:00,2,1,1,1,1,0.0,0.000000,0.0,...,2.0,2.0,-69.0,1.0,0.0,appldnld.apple.com,2,0.0,0.0,0.200000
2,10.10.10.11,2025-12-10 12:52:00,1,1,1,1,1,0.0,0.000000,0.0,...,1.0,1.0,0.0,0.0,0.0,connectivitycheck.gstatic.com,1,5.0,0.0,0.466558
3,10.10.10.11,2025-12-10 12:53:00,1,1,1,1,1,0.0,0.000000,0.0,...,2.0,2.0,0.0,0.0,0.0,connectivitycheck.gstatic.com,1,5.0,0.0,0.466558
4,10.10.10.11,2025-12-20 09:19:00,11,6,6,2,1,668.0,60.727273,167.0,...,8.0,4.0,334.0,5.0,0.0,connectivitycheck.gstatic.com,2,5.0,0.0,0.466558


,source.ip,window_start,req_count,uniq_dst_ips,uniq_domains,uniq_paths,uniq_dst_ports,resp_bytes_sum,resp_bytes_mean,resp_bytes_max,...,uniq_domains_roll_60m,uniq_paths_roll_60m,bytes_trend_30m,req_trend_30m,upload_trend_30m,http_beacon_best_domain,http_beacon_best_domain_req,http_beacon_best_persist_5m,http_beacon_best_regularity,http_beacon_score
50226,192.168.30.15,2025-11-22 13:50:00,9,1,1,9,1,1923.0,213.666667,880.0,...,117.0,358.0,0.344828,-1.482759,0.0,192.168.100.24,9,18.0,0.0,0.380879
50227,89.44.146.43,2025-11-25 12:37:00,1,1,1,1,1,0.0,0.000000,0.0,...,1.0,1.0,0.000000,0.000000,0.0,-,1,0.0,0.0,0.200000
50228,89.44.146.43,2025-11-25 13:18:00,1,1,1,1,1,0.0,0.000000,0.0,...,2.0,2.0,0.000000,0.000000,0.0,-,1,0.0,0.0,0.200000
50229,91.92.209.96,2025-11-25 19:45:00,1,1,1,1,1,0.0,0.000000,0.0,...,1.0,1.0,0.000000,0.000000,0.0,-,1,0.0,0.0,0.200000
50230,91.92.209.96,2025-11-25 21:54:00,1,1,1,1,1,0.0,0.000000,0.0,...,2.0,2.0,0.000000,0.000000,0.0,-,1,0.0,0.0,0.200000


In [19]:
# =========================
# Cell 4 — Train AutoEncoder on agg_features (HTTP)
# =========================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("🤖 Building AE dataset from agg_features...")

df_feat = agg_features.copy()

# ستون‌های غیرعددی/شناسه را حذف می‌کنیم
id_cols = ["source.ip", "window_start", "http_beacon_best_domain"]
X_df = df_feat.drop(columns=[c for c in id_cols if c in df_feat.columns], errors="ignore")

# فقط ستون‌های عددی
X_df = X_df.select_dtypes(include=[np.number]).replace([np.inf, -np.inf], 0.0).fillna(0.0)

print("✅ Numeric feature count:", X_df.shape[1])
print("✅ X shape:", X_df.shape)

# استانداردسازی
scaler_ae = StandardScaler()
X = scaler_ae.fit_transform(X_df.values).astype(np.float32)

# train/val split
X_train, X_val = train_test_split(X, test_size=0.2, random_state=42)

# مدل AE (ساده و سریع)
input_dim = X.shape[1]
enc_dim1 = max(32, input_dim // 2)
enc_dim2 = max(12, input_dim // 4)

inp = keras.Input(shape=(input_dim,))
x = layers.Dense(enc_dim1, activation="relu")(inp)
x = layers.Dropout(0.10)(x)
x = layers.Dense(enc_dim2, activation="relu")(x)
x = layers.Dense(enc_dim1, activation="relu")(x)
out = layers.Dense(input_dim, activation=None)(x)

ae = keras.Model(inp, out)
ae.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")

cb = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)
]

hist = ae.fit(
    X_train, X_train,
    validation_data=(X_val, X_val),
    epochs=80,
    batch_size=512,
    verbose=1,
    callbacks=cb
)

# reconstruction error = anomaly score
X_pred = ae.predict(X, batch_size=1024, verbose=0)
recon_err = np.mean((X_pred - X) ** 2, axis=1)

agg_features["ae_score"] = recon_err

print("✅ AE done. ae_score stats:")
print(agg_features["ae_score"].describe(percentiles=[0.9,0.95,0.99,0.995]))

# نمایش 10 مورد اول
cols_show = ["source.ip","window_start","ae_score","http_beacon_best_domain","http_beacon_score","req_count","resp_bytes_sum","req_body_bytes_sum","up_down_ratio_sum"]
cols_show = [c for c in cols_show if c in agg_features.columns]
display(agg_features.sort_values("ae_score", ascending=False)[cols_show].head(10))


🤖 Building AE dataset from agg_features...
✅ Numeric feature count: 49
✅ X shape: (50231, 49)
Epoch 1/80
79/79 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.9770 - val_loss: 0.7707
Epoch 2/80
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.6841 - val_loss: 0.4845
Epoch 3/80
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.4458 - val_loss: 0.3636
Epoch 4/80
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.3962 - val_loss: 0.3075
Epoch 5/80
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.3145 - val_loss: 0.2726
Epoch 6/80
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.2891 - val_loss: 0.2460
Epoch 7/80
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.2780 - val_loss: 0.2297
Epoch 8/80
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.2432 - val_loss: 0.2161
Epoch 9/80
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.2519 - val_loss: 0.2047
Epoch 10/80
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.2248 - val_loss: 0.1969
Epoch 11/80
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.2326

,source.ip,window_start,ae_score,http_beacon_best_domain,http_beacon_score,req_count,resp_bytes_sum,req_body_bytes_sum,up_down_ratio_sum
12763,172.16.18.4,2025-12-03 10:39:00,54.844856,192.168.14.2,0.327552,398,43504.0,3937.0,0.090495
12756,172.16.18.3,2025-12-03 10:51:00,37.337479,192.168.14.2,0.281747,5,101.0,0.0,0.000000
45660,192.168.20.17,2025-11-17 14:31:00,25.288256,149.154.165.120,0.386339,179,16912.0,51204.0,3.027494
9651,10.10.10.51,2025-11-30 06:48:00,22.141472,pl.hitgram.ir,0.438272,2726,6936.0,245632.0,35.408966
7892,10.10.10.39,2025-11-30 09:36:00,17.600094,(blank),0.550000,373,0.0,0.0,0.000000
9680,10.10.10.51,2025-11-30 07:18:00,16.693249,pl.hitgram.ir,0.417876,15,90.0,1409.0,15.483516
583,10.10.10.16,2025-11-16 13:21:00,12.969359,-,0.550000,78,1655.0,20106.0,12.141304
9648,10.10.10.51,2025-11-30 06:45:00,12.417194,pl.hitgram.ir,0.464868,1339,1470.0,28800.0,19.578518
8771,10.10.10.41,2025-11-16 12:26:00,11.010483,ctldl.windowsupdate.com,0.466558,95,4247.0,0.0,0.000000
12757,172.16.18.3,2025-12-03 10:53:00,10.081891,192.168.14.2,0.285427,1,15.0,340.0,21.250000


In [20]:
# =========================
# Cell 5 — LSTM Sequence Model (HTTP) + lstm_score  [FINAL FIXED + ROBUST]
#   ✅ جلوگیری از lstm_score_x / lstm_score_y
#   ✅ NaN را صفر نمی‌کنیم (تا pct خراب نشود)
#   ✅ split درستِ train/val (بدون قاطی شدن انتهای sequence ها)
#   ✅ جلوگیری از merge چندبه‌چند با groupby روی (ip, window_start)
#   ✅ seed برای تکرارپذیری
#   ✅ گزارش coverage + پیشنهاد اتومات برای SEQ_LEN
# =========================

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("🧠 Building sequences for HTTP LSTM (FINAL FIXED)...")

# ---- reproducibility ----
tf.random.set_seed(42)
np.random.seed(42)

# ---- work copy ----
df_seq = agg_features.copy()
df_seq = df_seq.sort_values(["source.ip", "window_start"]).reset_index(drop=True)

# --------- انتخاب فیچرهای تایم‌سری ----------
candidate_feats = [
    "req_count",
    "uniq_domains",
    "uniq_paths",
    "resp_bytes_sum",
    "req_body_bytes_sum",
    "up_down_ratio_sum",
    "http_beacon_score",
    "post_rate",
    "success_rate_2xx",
    "fail_rate_4xx5xx",
    "query_len_sum",
    "query_entropy_mean",
    "url_full_len_sum",
]
seq_feats = [c for c in candidate_feats if c in df_seq.columns]

if len(seq_feats) < 6:
    fallback = [c for c in ["req_count","resp_bytes_sum","req_body_bytes_sum","http_beacon_score","up_down_ratio_sum"] if c in df_seq.columns]
    seq_feats = list(dict.fromkeys(fallback + seq_feats))

print("✅ Using sequence features:", seq_feats)

# پاکسازی
df_seq[seq_feats] = (
    df_seq[seq_feats]
    .replace([np.inf, -np.inf], 0.0)
    .fillna(0.0)
)

# استانداردسازی
scaler_lstm = StandardScaler()
X_all = scaler_lstm.fit_transform(df_seq[seq_feats].values).astype(np.float32)

# --------- ساخت sequences ----------
SEQ_LEN = 20  # اگر coverage کم بود، 10 کن

seq_X, seq_y, seq_meta, seq_ip = [], [], [], []

for ip, g in df_seq.groupby("source.ip", sort=False):
    idx = g.index.values
    if len(idx) <= SEQ_LEN:
        continue

    X_ip = X_all[idx, :]
    for t in range(SEQ_LEN, len(idx)):
        seq_X.append(X_ip[t-SEQ_LEN:t, :])
        seq_y.append(X_ip[t, :])
        seq_meta.append((df_seq.loc[idx[t], "source.ip"], df_seq.loc[idx[t], "window_start"]))
        seq_ip.append(ip)

seq_X = np.array(seq_X, dtype=np.float32)
seq_y = np.array(seq_y, dtype=np.float32)
seq_ip = np.array(seq_ip)

print("✅ Sequences built:", seq_X.shape, "Targets:", seq_y.shape)

# اگر sequence ساخته نشد
if seq_X.shape[0] == 0:
    print("⚠️ No sequences built. Setting lstm_score to NaN for all rows.")
    for col in ["lstm_score", "lstm_score_x", "lstm_score_y"]:
        if col in agg_features.columns:
            agg_features = agg_features.drop(columns=[col])
    agg_features["lstm_score"] = np.nan

else:
    # --------- split train/val (بدون درهم‌ریختن تایم) ----------
    # به جای validation_split، خودمون split می‌کنیم تا کنترل داشته باشیم
    n = seq_X.shape[0]
    split = int(n * 0.8)

    X_train, y_train = seq_X[:split], seq_y[:split]
    X_val, y_val     = seq_X[split:], seq_y[split:]

    # --------- مدل LSTM ----------
    n_feat = seq_X.shape[2]

    inp = keras.Input(shape=(SEQ_LEN, n_feat))
    x = layers.LSTM(64, return_sequences=True)(inp)
    x = layers.LayerNormalization()(x)
    x = layers.Dropout(0.15)(x)
    x = layers.LSTM(32, return_sequences=False)(x)
    x = layers.LayerNormalization()(x)
    x = layers.Dense(32, activation="relu")(x)
    out = layers.Dense(n_feat, activation=None)(x)

    lstm = keras.Model(inp, out)
    lstm.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")

    cb = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    ]

    hist = lstm.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=25,
        batch_size=512,
        verbose=1,
        callbacks=cb,
        shuffle=False
    )

    # --------- محاسبه lstm_score ----------
    pred = lstm.predict(seq_X, batch_size=1024, verbose=0)
    err = np.mean((pred - seq_y) ** 2, axis=1).astype(np.float32)

    lstm_scores = pd.DataFrame(seq_meta, columns=["source.ip","window_start"])
    lstm_scores["lstm_score"] = err

    # ✅ جلوگیری از merge چندبه‌چند (اگر کلید تکراری شد)
    lstm_scores = lstm_scores.groupby(["source.ip","window_start"], as_index=False)["lstm_score"].mean()

    # ✅ حذف ستون‌های قدیمی برای جلوگیری از x/y
    for col in ["lstm_score", "lstm_score_x", "lstm_score_y"]:
        if col in agg_features.columns:
            agg_features = agg_features.drop(columns=[col])

    agg_features = agg_features.merge(lstm_scores, on=["source.ip","window_start"], how="left")

    # ✅ NaN را صفر نمی‌کنیم
    cov = 1.0 - float(agg_features["lstm_score"].isna().mean())
    print(f"✅ LSTM merged. Coverage (non-NaN lstm_score): {cov*100:.2f}%")

    # اگر coverage خیلی کم بود، پیشنهاد بده SEQ_LEN کم بشه
    if cov < 0.30:
        print("⚠️ Coverage is low. Suggestion: set SEQ_LEN = 10 and rerun Cell 5.")

    nz = agg_features.loc[agg_features["lstm_score"].notna(), "lstm_score"]
    print("✅ LSTM score stats (non-NaN rows):")
    print(nz.describe(percentiles=[0.9,0.95,0.99,0.995]))

    cols_show = ["source.ip","window_start","lstm_score","http_beacon_best_domain","http_beacon_score",
                 "req_count","resp_bytes_sum","req_body_bytes_sum","up_down_ratio_sum"]
    cols_show = [c for c in cols_show if c in agg_features.columns]
    display(agg_features.sort_values("lstm_score", ascending=False, na_position="last")[cols_show].head(15))

# چک سریع برای اطمینان از اینکه x/y نداریم
print("🔎 LSTM columns:", [c for c in agg_features.columns if "lstm_score" in c])


🧠 Building sequences for HTTP LSTM (FINAL FIXED)...
✅ Using sequence features: ['req_count', 'uniq_domains', 'uniq_paths', 'resp_bytes_sum', 'req_body_bytes_sum', 'up_down_ratio_sum', 'http_beacon_score', 'post_rate', 'success_rate_2xx', 'fail_rate_4xx5xx', 'query_len_sum', 'query_entropy_mean', 'url_full_len_sum']
✅ Sequences built: (49242, 20, 13) Targets: (49242, 13)
Epoch 1/25
77/77 ━━━━━━━━━━━━━━━━━━━━ 18s 162ms/step - loss: 1.8115 - val_loss: 0.5562
Epoch 2/25
77/77 ━━━━━━━━━━━━━━━━━━━━ 10s 132ms/step - loss: 1.3489 - val_loss: 0.5261
Epoch 3/25
77/77 ━━━━━━━━━━━━━━━━━━━━ 11s 145ms/step - loss: 1.2138 - val_loss: 0.5138
Epoch 4/25
77/77 ━━━━━━━━━━━━━━━━━━━━ 21s 147ms/step - loss: 1.1045 - val_loss: 0.5016
Epoch 5/25
77/77 ━━━━━━━━━━━━━━━━━━━━ 21s 149ms/step - loss: 1.0209 - val_loss: 0.4903
Epoch 6/25
77/77 ━━━━━━━━━━━━━━━━━━━━ 10s 136ms/step - loss: 0.9641 - val_loss: 0.4834
Epoch 7/25
77/77 ━━━━━━━━━━━━━━━━━━━━ 11s 143ms/step - loss: 0.9145 - val_loss: 0.4757
Epoch 8/25
77/77 ━

,source.ip,window_start,lstm_score,http_beacon_best_domain,http_beacon_score,req_count,resp_bytes_sum,req_body_bytes_sum,up_down_ratio_sum
9651,10.10.10.51,2025-11-30 06:48:00,869.257507,pl.hitgram.ir,0.438272,2726,6936.0,245632.0,35.408966
9659,10.10.10.51,2025-11-30 06:56:00,194.294266,pl.hitgram.ir,0.483947,1232,785.0,281471.0,358.105598
9654,10.10.10.51,2025-11-30 06:51:00,191.515228,pl.hitgram.ir,0.445228,1549,3104.0,119359.0,38.440902
9724,10.10.10.51,2025-11-30 08:03:00,176.233002,pl.hitgram.ir,0.424249,1109,4920.0,168422.0,34.225157
9648,10.10.10.51,2025-11-30 06:45:00,129.132172,pl.hitgram.ir,0.464868,1339,1470.0,28800.0,19.578518
7828,10.10.10.31,2025-12-20 10:36:00,104.804108,dl.delivery.mp.microsoft.com,0.509873,654,0.0,0.0,0.000000
9647,10.10.10.51,2025-11-30 06:44:00,87.536827,pl.hitgram.ir,0.459367,904,1166.0,25655.0,21.983719
46763,192.168.20.18,2025-12-10 12:56:00,81.845856,149.154.165.120,0.399446,675,14256.0,236960.0,16.620607
33363,192.168.20.13,2025-11-25 10:02:00,80.830063,feeds.elastic.co:443,0.550000,855,85896.0,11834.0,0.137770
4444,10.10.10.18,2025-12-10 10:33:00,79.586502,speed.cloudflare.com,0.550000,280,0.0,83700.0,83700.000000


🔎 LSTM columns: ['lstm_score']


In [21]:
# =========================
# Cell 6 — Fuse AE + LSTM + Beacon + Exfil → exact top 0.5% alerts (IMPROVED)
#   ✅ robust percentiles (log1p for heavy-tail)
#   ✅ dynamic AE/LSTM weights based on LSTM coverage
#   ✅ label thresholds computed on alerts for better separation
#   ✅ still EXACT top 0.5% alerts
# =========================

import numpy as np
import pandas as pd

print("🎯 Fusing AE + LSTM and calibrating severity (EXACT top 0.5%) [IMPROVED] ...")

scored = agg_features.copy()

# --------- helpers ----------
def safe_series(s):
    return pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)

def pct_rank(s: pd.Series) -> pd.Series:
    s = safe_series(s).fillna(0.0)
    return s.rank(pct=True, method="average")

def zscore(s: pd.Series) -> pd.Series:
    s = safe_series(s).fillna(0.0)
    mu = float(s.mean())
    sd = float(s.std()) + 1e-9
    return (s - mu) / sd

# ensure required cols exist
need_cols = ["ae_score", "lstm_score", "http_beacon_score",
             "req_body_bytes_sum", "up_down_ratio_sum",
             "req_count", "resp_bytes_sum"]
for c in need_cols:
    if c not in scored.columns:
        scored[c] = np.nan if c == "lstm_score" else 0.0

# ---------- robust scores (reduce heavy tail effect) ----------
# AE/LSTM ممکنه دُم سنگین داشته باشن → log1p بهتره
ae_raw = safe_series(scored["ae_score"]).fillna(0.0)
lstm_raw = safe_series(scored["lstm_score"])  # NaN allowed

scored["ae_log"] = np.log1p(ae_raw.clip(lower=0))
scored["lstm_log"] = np.log1p(lstm_raw.clip(lower=0))  # NaN stays NaN

# percentiles
scored["ae_pct"] = pct_rank(scored["ae_log"])

scored["lstm_pct"] = 0.0
mask_lstm = scored["lstm_log"].notna()
lstm_cov = float(mask_lstm.mean())
print(f"✅ LSTM coverage: {lstm_cov*100:.2f}%")

if mask_lstm.sum() > 10:
    scored.loc[mask_lstm, "lstm_pct"] = pct_rank(scored.loc[mask_lstm, "lstm_log"])
else:
    scored["lstm_pct"] = 0.0

scored["http_beacon_pct"] = pct_rank(scored["http_beacon_score"])

# ---------- exfil pressure (robust) ----------
rb = safe_series(scored["req_body_bytes_sum"]).fillna(0.0).clip(lower=0)
ud = safe_series(scored["up_down_ratio_sum"]).fillna(0.0).clip(lower=0)

scored["req_body_pct"] = pct_rank(np.log1p(rb))
scored["updown_pct"]   = pct_rank(np.log1p(ud))

# pressure ∈ [0..1]
scored["exfil_pressure"] = (0.65 * scored["req_body_pct"] + 0.35 * scored["updown_pct"]).clip(0, 1)

# ---------- dynamic AE/LSTM fusion ----------
# اگر coverage کم شد، LSTM وزنش کم بشه (اینجا تقریباً کامل)
w_lstm = 0.45 * min(1.0, max(0.0, lstm_cov / 0.80))  # cov>=80% → w_lstm≈0.45
w_ae   = 1.0 - w_lstm
print(f"✅ Weights → AE: {w_ae:.3f} | LSTM: {w_lstm:.3f}")

base = scored["ae_pct"].copy()
base.loc[mask_lstm] = (w_ae * scored.loc[mask_lstm, "ae_pct"] + w_lstm * scored.loc[mask_lstm, "lstm_pct"])
scored["fused_pct"] = base.clip(0, 1)

# ---------- controlled exfil boost (no saturation) ----------
ALPHA = 0.35
scored["final_pct"] = (scored["fused_pct"] + ALPHA * scored["exfil_pressure"] * (1.0 - scored["fused_pct"])).clip(0, 1)

# ---------- EXACT top 0.5% alerts ----------
ALERT_TOP_PCT = 0.005
N = len(scored)
K = int(np.ceil(N * ALERT_TOP_PCT))

scored["final_rank"] = scored["final_pct"].rank(method="first", ascending=False)
scored["is_alert"] = scored["final_rank"] <= K

# severity فقط برای alertها (1..99)
scored["severity"] = 0
if K > 1:
    sev = (1.0 - (scored.loc[scored["is_alert"], "final_rank"] - 1) / (K - 1)) * 98 + 1
else:
    sev = pd.Series([99], index=scored.index[scored["is_alert"]])

scored.loc[scored["is_alert"], "severity"] = sev.round().astype(int).clip(1, 99)

# ---------- labeling: thresholds computed on ALERTS (better) ----------
alerts = scored["is_alert"]
scored["alert_type"] = "BENIGN"

# اگر alert خیلی کم بود، fallback
if alerts.sum() > 5:
    # thresholds among alerts only
    PRESS_THR  = 0.90   # داخل alertها: top 10% pressure → exfil
    BEACON_THR = 0.92   # داخل alertها: top 8% beacon → beaconing

    press_thr_val  = scored.loc[alerts, "exfil_pressure"].quantile(PRESS_THR)
    beacon_thr_val = scored.loc[alerts, "http_beacon_pct"].quantile(BEACON_THR)

    # exfil first
    scored.loc[alerts & (scored["exfil_pressure"] >= press_thr_val), "alert_type"] = "HTTP_EXFIL_LOW_SLOW"

    # beaconing if not exfil-heavy
    scored.loc[
        alerts &
        (scored["http_beacon_pct"] >= beacon_thr_val) &
        (scored["exfil_pressure"] < press_thr_val),
        "alert_type"
    ] = "HTTP_BEACONING"

    # remaining
    scored.loc[alerts & (scored["alert_type"] == "BENIGN"), "alert_type"] = "OTHER_ANOMALY"
else:
    scored.loc[alerts, "alert_type"] = "OTHER_ANOMALY"

# ---------- debug helpers (optional) ----------
scored["ae_z"] = zscore(scored["ae_log"])
scored["lstm_z"] = 0.0
scored.loc[mask_lstm, "lstm_z"] = zscore(scored.loc[mask_lstm, "lstm_log"])

# ---------- quick report ----------
print("✅ Total rows:", N)
print(f"✅ Alerts: {alerts.sum()} ({alerts.mean()*100:.3f}%) target≈{ALERT_TOP_PCT*100:.1f}%")
print("✅ Split (alerts):")
print(scored.loc[alerts, "alert_type"].value_counts())

print("\n✅ lstm_pct stats (rows with LSTM):")
if mask_lstm.sum() > 0:
    print(scored.loc[mask_lstm, "lstm_pct"].describe())

cols_show = [
    "source.ip","window_start","severity","alert_type",
    "ae_score","lstm_score","ae_pct","lstm_pct","fused_pct","final_pct",
    "http_beacon_best_domain","http_beacon_score","http_beacon_pct",
    "req_count","resp_bytes_sum","req_body_bytes_sum","up_down_ratio_sum","exfil_pressure",
    "final_rank"
]
cols_show = [c for c in cols_show if c in scored.columns]
display(scored.sort_values(["severity","final_pct"], ascending=False)[cols_show].head(25))

OUT1 = "http_ml_scored_severity.csv"
scored.to_csv(OUT1, index=False)
print(f"\n📦 Saved: {OUT1}")

http_scored = scored.copy()


🎯 Fusing AE + LSTM and calibrating severity (EXACT top 0.5%) [IMPROVED] ...
✅ LSTM coverage: 98.03%
✅ Weights → AE: 0.550 | LSTM: 0.450
✅ Total rows: 50231
✅ Alerts: 252 (0.502%) target≈0.5%
✅ Split (alerts):
alert_type
OTHER_ANOMALY          205
HTTP_EXFIL_LOW_SLOW     26
HTTP_BEACONING          21
Name: count, dtype: int64

✅ lstm_pct stats (rows with LSTM):
count    49242.000000
mean         0.500010
std          0.288678
min          0.000020
25%          0.250015
50%          0.500010
75%          0.750005
max          1.000000
Name: lstm_pct, dtype: float64


,source.ip,window_start,severity,alert_type,ae_score,lstm_score,ae_pct,lstm_pct,fused_pct,final_pct,http_beacon_best_domain,http_beacon_score,http_beacon_pct,req_count,resp_bytes_sum,req_body_bytes_sum,up_down_ratio_sum,exfil_pressure,final_rank
12763,172.16.18.4,2025-12-03 10:39:00,99,OTHER_ANOMALY,54.844856,NaN,1.000000,0.000000,1.000000,1.000000,192.168.14.2,0.327552,0.004698,398,43504.0,3937.0,0.090495,0.629262,1.0
12756,172.16.18.3,2025-12-03 10:51:00,99,OTHER_ANOMALY,37.337479,NaN,0.999980,0.000000,0.999980,0.999981,192.168.14.2,0.281747,0.003783,5,101.0,0.0,0.000000,0.181103,2.0
9651,10.10.10.51,2025-11-30 06:48:00,98,OTHER_ANOMALY,22.141472,869.257507,0.999940,1.000000,0.999967,0.999978,pl.hitgram.ir,0.438272,0.527324,2726,6936.0,245632.0,35.408966,0.980867,3.0
45660,192.168.20.17,2025-11-17 14:31:00,98,OTHER_ANOMALY,25.288256,NaN,0.999960,0.000000,0.999960,0.999973,149.154.165.120,0.386339,0.304035,179,16912.0,51204.0,3.027494,0.924198,4.0
7892,10.10.10.39,2025-11-30 09:36:00,97,OTHER_ANOMALY,17.600094,NaN,0.999920,0.000000,0.999920,0.999925,(blank),0.550000,0.603691,373,0.0,0.0,0.000000,0.181103,5.0
9648,10.10.10.51,2025-11-30 06:45:00,97,OTHER_ANOMALY,12.417194,129.132172,0.999861,0.999919,0.999887,0.999925,pl.hitgram.ir,0.464868,0.532699,1339,1470.0,28800.0,19.578518,0.967634,6.0
583,10.10.10.16,2025-11-16 13:21:00,97,OTHER_ANOMALY,12.969359,NaN,0.999881,0.000000,0.999881,0.999920,-,0.550000,0.827348,78,1655.0,20106.0,12.141304,0.949676,7.0
9654,10.10.10.51,2025-11-30 06:51:00,96,OTHER_ANOMALY,7.099233,191.515228,0.999721,0.999959,0.999828,0.999887,pl.hitgram.ir,0.445228,0.528518,1549,3104.0,119359.0,38.440902,0.980896,8.0
12757,172.16.18.3,2025-12-03 10:53:00,96,OTHER_ANOMALY,10.081891,NaN,0.999821,0.000000,0.999821,0.999856,192.168.14.2,0.285427,0.003822,1,15.0,340.0,21.250000,0.565037,9.0
8771,10.10.10.41,2025-11-16 12:26:00,95,OTHER_ANOMALY,11.010483,NaN,0.999841,0.000000,0.999841,0.999851,ctldl.windowsupdate.com,0.466558,0.533655,95,4247.0,0.0,0.000000,0.181103,10.0



📦 Saved: http_ml_scored_severity.csv


In [22]:
# =========================
# Cell 7 — Enrich HTTP alerts with raw context + download (IMPROVED + FAST)
# Output: http_ml_scored_severity_enriched.csv
# =========================

import pandas as pd
import numpy as np
from google.colab import files

print("🧩 Enriching HTTP alerts with raw context (IMPROVED + FAST)...")

# ---------- helpers ----------
def mode_safe(s):
    if s is None or len(s) == 0:
        return ""
    vc = s.value_counts(dropna=True)
    return str(vc.index[0]) if len(vc) else ""

def topk_str(s, k=3, sep=" | "):
    """Return top-k most frequent values as single string."""
    if s is None or len(s) == 0:
        return ""
    vc = s.value_counts(dropna=True).head(k)
    if len(vc) == 0:
        return ""
    return sep.join([f"{str(idx)}({int(cnt)})" for idx, cnt in vc.items()])

def trunc(s, n=120):
    s = "" if s is None else str(s)
    return s if len(s) <= n else s[:n] + "…"

# ---------- pick scored dataframe safely ----------
# اولویت: http_scored (خروجی Cell 6)، بعد scored، بعد فایل CSV
if "http_scored" in globals():
    df_out = http_scored.copy()
elif "scored" in globals():
    df_out = scored.copy()
else:
    df_out = pd.read_csv("http_ml_scored_severity.csv")

# sanity
if "severity" not in df_out.columns:
    raise ValueError("⛔ severity column not found in scored file/dataframe. Run Cell 6 properly.")

# ---------- normalize join keys ----------
df_out["source.ip"] = df_out["source.ip"].astype(str).str.strip()
df_out["window_start"] = pd.to_datetime(df_out["window_start"], errors="coerce").dt.floor("min")

df_clean["source.ip"] = df_clean["source.ip"].astype(str).str.strip()
df_clean["window_start"] = pd.to_datetime(df_clean["window_start"], errors="coerce").dt.floor("min")

# Alerts only (keys)
alerts_keys = (
    df_out.loc[df_out["severity"] > 0, ["source.ip", "window_start"]]
    .dropna()
    .drop_duplicates()
)
print("✅ Alerts to enrich:", len(alerts_keys))

if len(alerts_keys) == 0:
    print("⚠️ No alerts found (severity>0). Saving scored as-is.")
    OUT = "http_ml_scored_severity_enriched.csv"
    df_out.to_csv(OUT, index=False)
    files.download(OUT)
    raise SystemExit()

# ---------- FAST filter df_clean to alert windows only ----------
# Trick: build a join key string to filter quickly
alerts_keys["_k"] = alerts_keys["source.ip"].astype(str) + "||" + alerts_keys["window_start"].astype(str)

df_clean["_k"] = df_clean["source.ip"].astype(str) + "||" + df_clean["window_start"].astype(str)

raw_alert = df_clean[df_clean["_k"].isin(set(alerts_keys["_k"]))].copy()
print("✅ Raw rows matched for alerts:", len(raw_alert))

# cleanup helper column
raw_alert.drop(columns=["_k"], inplace=True, errors="ignore")
df_clean.drop(columns=["_k"], inplace=True, errors="ignore")
alerts_keys.drop(columns=["_k"], inplace=True, errors="ignore")

# If still 0, show diagnosis
if len(raw_alert) == 0:
    print("⚠️ No matches found. Diagnosis:")
    print("  df_out alert key sample:", df_out.loc[df_out["severity"] > 0, ["source.ip","window_start"]].head(5).to_dict("records"))
    print("  df_clean key sample:", df_clean[["source.ip","window_start"]].head(5).to_dict("records"))
    # Save anyway
    OUT = "http_ml_scored_severity_enriched.csv"
    df_out.to_csv(OUT, index=False)
    print("📦 Saved (no enrichment):", OUT)
    files.download(OUT)
    raise SystemExit()

# ---------- build context ----------
raw_alert["is_post"] = (raw_alert["http.method"].astype(str).str.upper() == "POST").astype(int)
raw_alert["is_get"]  = (raw_alert["http.method"].astype(str).str.upper() == "GET").astype(int)

# truncate long fields to keep CSV sane
raw_alert["url.query"] = raw_alert["url.query"].astype(str).apply(lambda x: trunc(x, 160))
raw_alert["url.full"]  = raw_alert["url.full"].astype(str).apply(lambda x: trunc(x, 220))

ctx = raw_alert.groupby(["source.ip", "window_start"]).agg(
    raw_rows=("ts", "count"),

    top_domain=("url.domain", mode_safe),
    top_path=("url.path", mode_safe),
    top_query=("url.query", mode_safe),
    top_dst_ip=("destination.ip", mode_safe),
    top_dst_port=("destination.port", mode_safe),

    top_method=("http.method", mode_safe),
    top_status=("http.status", mode_safe),
    top_http_version=("http.version", mode_safe),

    # top-3 lists (very useful)
    top3_domains=("url.domain", lambda s: topk_str(s, k=3)),
    top3_paths=("url.path", lambda s: topk_str(s, k=3)),
    top3_dsts=("destination.ip", lambda s: topk_str(s, k=3)),
    top3_status=("http.status", lambda s: topk_str(s, k=3)),

    resp_bytes_sum=("resp_bytes", "sum"),
    resp_bytes_mean=("resp_bytes", "mean"),
    resp_bytes_max=("resp_bytes", "max"),

    req_body_bytes_sum=("req_body_bytes", "sum"),
    req_body_bytes_mean=("req_body_bytes", "mean"),
    req_body_bytes_max=("req_body_bytes", "max"),

    success_rate_raw=("is_success", "mean"),
    fail_rate_raw=("is_fail", "mean"),
    post_rate_raw=("is_post", "mean"),
    get_rate_raw=("is_get", "mean"),

    uniq_domains_raw=("url.domain", pd.Series.nunique),
    uniq_paths_raw=("url.path", pd.Series.nunique),
    uniq_dst_ips_raw=("destination.ip", pd.Series.nunique),

    query_len_sum=("url.query", lambda s: s.astype(str).str.len().sum()),
    url_full_len_sum=("url.full", lambda s: s.astype(str).str.len().sum()),
).reset_index()

# Human-readable context string (compact)
ctx["alert_context"] = (
    "top_domain=" + ctx["top_domain"].astype(str) +
    " | top_path=" + ctx["top_path"].astype(str) +
    " | top_dst=" + ctx["top_dst_ip"].astype(str) + ":" + ctx["top_dst_port"].astype(str) +
    " | method=" + ctx["top_method"].astype(str) +
    " | status=" + ctx["top_status"].astype(str) +
    " | rows=" + ctx["raw_rows"].astype(str) +
    " | resp_sum=" + ctx["resp_bytes_sum"].round(0).astype(int).astype(str) +
    " | up_sum=" + ctx["req_body_bytes_sum"].round(0).astype(int).astype(str) +
    " | succ=" + (100*ctx["success_rate_raw"]).round(1).astype(str) + "%" +
    " | fail=" + (100*ctx["fail_rate_raw"]).round(1).astype(str) + "%" +
    " | post=" + (100*ctx["post_rate_raw"]).round(1).astype(str) + "%"
)

# ---------- merge back ----------
df_enriched = df_out.merge(ctx, on=["source.ip", "window_start"], how="left")

OUT = "http_ml_scored_severity_enriched.csv"
df_enriched.to_csv(OUT, index=False)
print("📦 Saved:", OUT)

# Show top alerts enriched
display(
    df_enriched[df_enriched["severity"] > 0]
    .sort_values(["severity","final_pct"], ascending=False)
    .head(25)
)

files.download(OUT)


🧩 Enriching HTTP alerts with raw context (IMPROVED + FAST)...
✅ Alerts to enrich: 252
✅ Raw rows matched for alerts: 89747
📦 Saved: http_ml_scored_severity_enriched.csv


,source.ip,window_start,req_count,uniq_dst_ips,uniq_domains,uniq_paths,uniq_dst_ports,resp_bytes_sum_x,resp_bytes_mean_x,resp_bytes_max_x,...,success_rate_raw,fail_rate_raw,post_rate_raw,get_rate_raw,uniq_domains_raw,uniq_paths_raw,uniq_dst_ips_raw,query_len_sum_y,url_full_len_sum_y,alert_context
12763,172.16.18.4,2025-12-03 10:39:00,398,1,2,308,1,43504.0,109.306533,972.0,...,0.984925,0.002513,0.120603,0.851759,2.0,308.0,1.0,1323.0,27636.0,top_domain=192.168.14.2 | top_path=/api/conten...
12756,172.16.18.3,2025-12-03 10:51:00,5,1,1,3,1,101.0,20.200000,86.0,...,0.800000,0.000000,0.400000,0.600000,1.0,3.0,1.0,5.0,170.0,top_domain=192.168.14.2 | top_path=/api/spaces...
9651,10.10.10.51,2025-11-30 06:48:00,2726,1,1,2726,1,6936.0,2.544387,858.0,...,0.998533,0.000734,0.959648,0.040352,1.0,2726.0,1.0,2726.0,308592.0,top_domain=pl.hitgram.ir | top_path=/---@MiTiV...
45660,192.168.20.17,2025-11-17 14:31:00,179,1,4,2,1,16912.0,94.480447,968.0,...,1.000000,0.000000,0.994413,0.000000,4.0,2.0,1.0,179.0,3288.0,top_domain=149.154.165.120 | top_path=/api | t...
7892,10.10.10.39,2025-11-30 09:36:00,373,2,2,2,2,0.0,0.000000,0.0,...,0.501340,0.000000,0.000000,1.000000,2.0,2.0,2.0,373.0,3018.0,top_domain=(blank) | top_path=/ | top_dst=78.1...
9648,10.10.10.51,2025-11-30 06:45:00,1339,1,1,1339,1,1470.0,1.097834,858.0,...,0.999253,0.000747,0.974608,0.025392,1.0,1339.0,1.0,1339.0,151718.0,top_domain=pl.hitgram.ir | top_path=/---@MiTiV...
583,10.10.10.16,2025-11-16 13:21:00,78,25,4,2,1,1655.0,21.217949,740.0,...,0.243590,0.000000,0.961538,0.012821,4.0,2.0,25.0,78.0,1606.0,top_domain=speed.cloudflare.com | top_path=/ |...
9654,10.10.10.51,2025-11-30 06:51:00,1549,1,1,1549,1,3104.0,2.003873,467.0,...,0.991607,0.000000,0.954810,0.045190,1.0,1549.0,1.0,1549.0,175106.0,top_domain=pl.hitgram.ir | top_path=/---@MiTiV...
12757,172.16.18.3,2025-12-03 10:53:00,1,1,1,1,1,15.0,15.000000,15.0,...,1.000000,0.000000,1.000000,0.000000,1.0,1.0,1.0,1.0,36.0,top_domain=192.168.14.2 | top_path=/api/ui_cou...
8771,10.10.10.41,2025-11-16 12:26:00,95,7,6,8,1,4247.0,44.705263,958.0,...,0.926316,0.063158,0.000000,1.000000,6.0,8.0,7.0,13616.0,8067.0,top_domain=185.40.106.198 | top_path=/filestre...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>